In [ ]:
# 三个聊天机器人的讨论（GPT + Ollama + Gemini）
# Week 2 Day 1 风格：Alex 抬杠 / Blake 附和 / Charlie 冷静调解，轮流辩论一个话题


In [ ]:
# ========== 导入 ==========
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 导入标准库 os：getenv 读 OLLAMA_API_KEY、GEMINI_API_KEY
import os
# 从 openai 导入 OpenAI：统一 Chat Completions 客户端
from openai import OpenAI
# 从 IPython.display 导入 display、Markdown：笔记本里展示对话
from IPython.display import display, Markdown


In [ ]:
# ========== 配置：环境 + 三个 OpenAI 兼容客户端 + 模型名 ==========
# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 本地 Ollama 的 OpenAI 兼容 base_url（URL 原样保留）
ollama_api_url = 'http://localhost:11434/v1'
# Gemini 的 OpenAI 兼容 base_url（URL 原样保留）
gemini_api_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'

# 默认 OpenAI 客户端（读 OPENAI_API_KEY）
openai = OpenAI()
# Ollama 客户端：密钥来自环境变量 OLLAMA_API_KEY，base_url 指向本机
ollama = OpenAI(api_key=os.getenv('OLLAMA_API_KEY'), base_url=ollama_api_url)
# Gemini 客户端：密钥来自 GEMINI_API_KEY，base_url 指向 Google 兼容层
gemini = OpenAI(api_key=os.getenv('GEMINI_API_KEY'), base_url=gemini_api_url)

# 三个模型 id（字符串必须与后端认识的名字一致，勿改）
gpt_model = 'gpt-4.1-mini'
ollama_model = 'mistral-nemo'
gemini_model = 'models/gemini-2.5-flash'


In [ ]:
# ========== 提示词：共享历史 + 三人 system/user 模板（英文 prompt 原样）==========
# 对话历史：元素是 {角色名: 文本} 的字典
conversation = []

# Alex（GPT）人设：好辩、抬杠
system_prompt_alex = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie. Write only your respond as Alex don't add anything unnecessary.
"""

# Alex 的 user 模板：注入 {conversation} 历史
user_prompt_alex_template = """
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""

# Blake（Gemini）人设：礼貌、附和
system_prompt_blake = """
You are Blake, a chatbot who is very polite; you agree with anything in the conversation and you challenge everything, in a nice way.
You are in a conversation with Blake and Charlie. Write only your respond as Blake don't add anything unnecessary.
"""
# Blake 的 user 模板
user_prompt_blake_template = """
You are Blake, in conversation with Alex and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Blake.
"""
# Charlie（Ollama）人设：冷静中立的调解者
system_prompt_charlie = """
You are Charlie, a chatbot who is strictly clinical, objective, and focused on facts.
You act as the mediator between Alex and Blake.
Your goal is to remain completely neutral and detached from their emotional extremes.
When Alex is snarky or Blake is overly agreeable, you point out the logical inconsistencies
or redirect the conversation back to the original topic with dry, analytical precision.
You are in a conversation with Alex and Blake. Write only your respond as Charlie don't add anything unnecessary. Don't write your name like 'Charlie: Hello' Just say 'Hello' Dont add unnecessary quotation marks.
"""
# Charlie 的 user 模板
user_prompt_charlie_template = """
You are Charlie, in conversation with Alex and Blake.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Charlie.
"""


In [ ]:
# ========== 格式化历史 + 调用 GPT（Alex）==========
def format_conversation():
    # 把 conversation 列表拼成 "Name: text\n" 多行字符串，供模板填充
    formatted = ""
    for msg in conversation:
        # 每个 msg 是单键字典：角色名 -> 发言内容
        for name, text in msg.items():
            formatted += f"{name}: {text}\n"
    return formatted

def call_gpt():
    # 用模板 + 当前历史生成 Alex 的 user prompt
    user_prompt = user_prompt_alex_template.format(conversation=format_conversation())
    # 非流式 Chat Completions：system=人设，user=带历史提示
    response = openai.chat.completions.create(model=gpt_model, messages=[{"role": "system", "content": system_prompt_alex} , {"role": "user", "content": user_prompt}])
    # 包装成 {"Alex": 回复}，追加进共享历史
    message = {"Alex": response.choices[0].message.content}
    conversation.append(message)
    return message


In [ ]:
# ========== 调用 Ollama（Charlie）==========
def call_ollama():
    # Charlie 用 charlie 模板；客户端是本地 ollama
    user_prompt = user_prompt_charlie_template.format(conversation=format_conversation())
    # model / system / user 均对应 Charlie 角色
    response = ollama.chat.completions.create(model=ollama_model, messages=[{"role": "system", "content": system_prompt_charlie} , {"role": "user", "content": user_prompt}])
    # 写入共享 conversation 后返回本轮消息
    message = {"Charlie": response.choices[0].message.content}
    conversation.append(message)
    return message


In [ ]:
# ========== 调用 Gemini（Blake）==========
def call_gemini():
    # Blake 用 blake 模板；客户端是 gemini 兼容层
    user_prompt = user_prompt_blake_template.format(conversation=format_conversation())
    # 非流式一次拿完整回复
    response = gemini.chat.completions.create(model=gemini_model, messages=[{"role": "system", "content": system_prompt_blake} , {"role": "user", "content": user_prompt}])
    # 追加进共享历史并返回
    message = {"Blake": response.choices[0].message.content}
    conversation.append(message)
    return message


In [ ]:
# ========== 主对话：设定辩题后轮流发言 ==========
# 想换话题时，只改下面 Moderator 字符串里的英文辩题即可（这是发给模型的内容，保持英文）
conversation = [{"Moderator": "Hello everyone! Today's topic is: Are hotdogs sandwiches? Let's start the debate."}]
# 先展示主持人开场
display(Markdown(f"**Moderator:** {conversation[0]['Moderator']}"))

# 5 轮：Alex → Charlie → Blake
for _ in range(5):
    # Alex（GPT）
    alex = call_gpt()
    display(Markdown(f"**Alex:** {alex['Alex']}"))
    # Charlie（Ollama）
    charlie = call_ollama()
    display(Markdown(f"**Charlie:** {charlie['Charlie']}"))
    # Blake（Gemini）
    blake = call_gemini()
    display(Markdown(f"**Blake:** {blake['Blake']}"))



In [ ]:
# （空单元格）可在此继续实验：例如加流式输出、换辩题、改轮次
